---

# Gera Netcdf de Quantidade de Dias Secos e Temperatura Máxima no Mês de Agosto de 2024 Acima do Percentil de 90% com Dados do CPC.

---

- `REALIZADO POR`:
> Prof. Enrique V. Mattos / UNIFEI - 29/05/2026

- `ATUALIZADO POR`:
> Prof. Enrique V. Mattos / UNIFEI - 29/05/2026
---

# Prepando ambiente


In [ ]:
#=========================================================================================================================#
#                                          INSTALAÇÃO E IMPORTAÇÃO DAS BIBLIOTECAS
#=========================================================================================================================#
# instala bibliotecas
!pip install -q xarray dask netCDF4 bottleneck xclim ultraplot cartopy salem rasterio pyproj geopandas

# importa bibliotecas
import xarray as xr
import xclim as xc
import ultraplot as uplt
import salem
import os
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs
import numpy as np
from dask.diagnostics import ProgressBar
import warnings
warnings.filterwarnings("ignore")

#=========================================================================================================================#
#                                  MONTA O GOOGLE DRIVE E CRIA O DIRETÓRIO DE SAÍDA
#=========================================================================================================================#
# monta o drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/2-PESQUISA/artigo_queimadas_Paola_2026/0_round1_revisoes/dias_secos_e_diasacimaP90_e_focoscalor'

#=========================================================================================================================#
#                                  DEFINE OS LIMITES DO BRASIL OU ESTADO DE SÃO PAULO
#=========================================================================================================================#
# limites
#lonmin, lonmax, latmin, latmax = -53.3, -43.9, -25.4, -19.7 # estado de SP
lonmin, lonmax, latmin, latmax = -75.0, -33.0, -35.0, 7.0 # Brasil

# leitura shapefiles
#shapefile_regiao = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/UFs/SP/SP_UF_2019.shp') # estado de SP
shapefile_regiao = salem.read_shapefile('https://github.com/evmpython/shapefile/raw/main/brasil/BRAZIL.shp') # Brasil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 384.7/384.7 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.5/83.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.4 MB/s eta 0:00:00


Mounted at /content/drive


# Calcula Quantidade de Dias Secos

In [ ]:
#=========================================================================================================================#
#                                            CARREGA OS DADOS DO CPC
#=========================================================================================================================#
# carrega os dados
cpc_precip_diaria = xr.open_dataset('http://apdrc.soest.hawaii.edu:80/dods/public_data/Interpolated_precipitation/cpc_rainfall/Daily').squeeze()

# transforma as longitudes de 0/360 para -180/+180
cpc_precip_diaria.coords['lon'] = ((cpc_precip_diaria.coords['lon'] + 180) % 360) - 180 ; cpc_precip_diaria = cpc_precip_diaria.sortby(cpc_precip_diaria.lon)

# datas
data_inicial, data_final = '2024-08-01', '2024-08-31'
data_inicial, data_final = '2024-09-01', '2024-09-30'

# seleciona os dados
cpc_precip_diaria = cpc_precip_diaria.sel(lon=slice(lonmin,lonmax), lat=slice(latmin,latmax)).sel(time=slice(data_inicial, data_final))

#=========================================================================================================================#
#                                         CALCULA A QUANTIDADE DE DIAS SECOS
#=========================================================================================================================#
# atribuição de unidade
cpc_precip_diaria['precip'].attrs['units'] = 'mm/d'

# número de dias secos do mês de agosto de 2024
dry_days_cpc = xc.indices.dry_days(cpc_precip_diaria['precip'], thresh='1.0 mm/d', freq='YS')

# recorta para o Brasil
#cpc_precip_diaria = cpc_precip_diaria.salem.roi(shape=shapefile_regiao)
#dry_days_cpc = dry_days_cpc.salem.roi(shape=shapefile_regiao)

# salva netcdf
dry_days_cpc.to_netcdf(f'{dir}/dry_days_cpc_{data_inicial}_{data_final}.nc')

In [ ]:
dry_days_cpc = xc.indices.dry_days(cpc_precip_diaria['precip'], thresh='1.0 mm/d', freq='YS')

tx_days_above = xc.indices.tx_days_above(cpc_tmax_diaria['tmax'], thresh='35 C', freq='YS')

# https://xclim.readthedocs.io/en/stable/_modules/xclim/indices/_multivariate.html#tx90p
# https://xclim.readthedocs.io/en/stable/indices.html#xclim.indices.tx90p

# Calcula Quantidade de Dias com Temperatura Máxima > P90

In [ ]:
%%time
#=========================================================================================================================#
#                                                CARREGA OS DADOS DO CPC
#=========================================================================================================================#
mes_num, mes_nome, dia_final_mes = '08', 'agosto', '31'
mes_num, mes_nome, dia_final_mes = '09', 'setembro', '30'

#=========================================================================================================================#
#                                                CARREGA OS DADOS DO CPC
#=========================================================================================================================#
print("[1/7] Carregando dados do CPC...")

# carrega com chunking para processamento lazy
cpc_tmax_diaria = xr.open_dataset('http://apdrc.soest.hawaii.edu:80/dods/public_data/CPC_Temperature/tmax', chunks={'time': 100, 'lat': 50, 'lon': 50}).squeeze()

# transforma longitudes
cpc_tmax_diaria.coords['lon'] = ((cpc_tmax_diaria.coords['lon'] + 180) % 360) - 180 ; cpc_tmax_diaria = cpc_tmax_diaria.sortby(cpc_tmax_diaria.lon)

# recorta o dado para regiao de estudo
cpc_tmax_diaria = cpc_tmax_diaria.sel(lon=slice(lonmin, lonmax), lat=slice(latmin, latmax))

# atribui unidade
cpc_tmax_diaria['tmax'].attrs['units'] = 'C'

#=========================================================================================================================#
#                                                 CÁLCULO DO P90
#=========================================================================================================================#
hist_start_year, hist_end_year = 1980, 2024

print(f"[2/7]: Calculando P90 para {mes_nome} {hist_start_year}-{hist_end_year}...")

# calcula o percentil mensal diretamente
with ProgressBar():

    # extrai apenas os dados necessários (agostos/setembros)
    aug_months = []
    for year in range(hist_start_year, hist_end_year + 1):
        print(f"  Processando {mes_nome} de {year}...")

        # Modified to select all days of the month, not just the first day
        aug_data = cpc_tmax_diaria['tmax'].sel(time=slice(f'{year}-{mes_num}-01', f'{year}-{mes_num}-{dia_final_mes}'))
        aug_months.append(aug_data)

    # empilha todos os agostos/setembros
    cpc_tmax_aug_hist = xr.concat(aug_months, dim='time')

    # calcula percentil
    tmax_p90 = cpc_tmax_aug_hist.quantile(0.90, dim='time', keep_attrs=True).compute()
    tmax_p90.attrs['units'] = 'C'

print("[3/7]: P90 calculado!")

#=========================================================================================================================#
#                             CALCULA A QUANTIDADE DE DIAS ACIMA DO P90 EM AGOSTO ou SETEMBRO/2024
#=========================================================================================================================#
print(f"[4/7]: Processando {mes_nome} de 2024...")

# carrega apenas agosto/setembro de 2024
tmax_aug_2024 = cpc_tmax_diaria['tmax'].sel(time=slice(f'2024-{mes_num}-01', f'2024-{mes_num}-{dia_final_mes}')).compute()
tmax_aug_2024.attrs['units'] = 'C'

print("[5/7]: Calculando dias acima do P90...")

# contagem de dias
dias_acima_p90 = (tmax_aug_2024 > tmax_p90).sum(dim='time').astype(int)
dias_acima_p90.name = f'dias_acima_p90_{mes_nome}_2024'

print("[6/7]: Aplicando máscara do Brasil...")
#dias_acima_p90 = dias_acima_p90.salem.roi(shape=shapefile_regiao).compute()

#=========================================================================================================================#
#                                            SALVA RESULTADO ARQUIVO NETCDF
#=========================================================================================================================#

print("[7/7]: Salvando arquivo...")
output_filename = f"{dir}/dias_tmax_acima_p90_{mes_num}_2024.nc"
dias_acima_p90.to_netcdf(output_filename)

print(f"\n✅ Arquivo salvo: {output_filename}")
print(f"📊 Total de dias (média espacial): {dias_acima_p90.mean().values:.1f} dias")
print(f"📍 Máximo local: {dias_acima_p90.max().values} dias")
print(f"📍 Mínimo local: {dias_acima_p90.min().values} dias")

[1/7] Carregando dados do CPC...
[2/7]: Calculando P90 para setembro 1980-2024...
  Processando setembro de 1980...
  Processando setembro de 1981...
  Processando setembro de 1982...
  Processando setembro de 1983...
  Processando setembro de 1984...
  Processando setembro de 1985...
  Processando setembro de 1986...
  Processando setembro de 1987...
  Processando setembro de 1988...
  Processando setembro de 1989...
  Processando setembro de 1990...
  Processando setembro de 1991...
  Processando setembro de 1992...
  Processando setembro de 1993...
  Processando setembro de 1994...
  Processando setembro de 1995...
  Processando setembro de 1996...
  Processando setembro de 1997...
  Processando setembro de 1998...
  Processando setembro de 1999...
  Processando setembro de 2000...
  Processando setembro de 2001...
  Processando setembro de 2002...
  Processando setembro de 2003...
  Processando setembro de 2004...
  Processando setembro de 2005...
  Processando setembro de 2006...


In [ ]:
# todos agostos como dataset do xarray
cpc_tmax_aug_hist

<xarray.DataArray 'tmax' (time: 1350, lat: 84, lon: 84)> Size: 38MB
dask.array<concatenate, shape=(1350, 84, 84), dtype=float32, chunksize=(30, 44, 48), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) datetime64[ns] 11kB 1980-09-01 1980-09-02 ... 2024-09-30
  * lat      (lat) float64 672B -34.75 -34.25 -33.75 -33.25 ... 5.75 6.25 6.75
  * lon      (lon) float64 672B -74.75 -74.25 -73.75 ... -34.25 -33.75 -33.25
Attributes:
    long_name:  daily maximum temperature [deg c] 
    units:      C

In [ ]:
# percentil
tmax_p90

<xarray.DataArray 'tmax' (lat: 84, lon: 84)> Size: 28kB
array([[      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       [      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       [      nan,       nan,       nan, ...,       nan,       nan,
              nan],
       ...,
       [34.97929 , 31.770168, 22.321054, ...,       nan,       nan,
              nan],
       [33.668518, 34.323193, 29.348207, ...,       nan,       nan,
              nan],
       [31.503536, 34.215252, 33.68714 , ...,       nan,       nan,
              nan]], dtype=float32)
Coordinates:
  * lat       (lat) float64 672B -34.75 -34.25 -33.75 -33.25 ... 5.75 6.25 6.75
  * lon       (lon) float64 672B -74.75 -74.25 -73.75 ... -34.25 -33.75 -33.25
    quantile  float64 8B 0.9
Attributes:
    long_name:  daily maximum temperature [deg c] 
    units:      C

In [ ]:
# agosto/2024
tmax_aug_2024

<xarray.DataArray 'tmax' (time: 30, lat: 84, lon: 84)> Size: 847kB
array([[[      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        ...,
        [35.11214 , 32.53908 , 23.413176, ...,       nan,       nan,
               nan],
        [33.773487, 34.977913, 30.135572, ...,       nan,       nan,
               nan],
        [31.691664, 34.680424, 34.038948, ...,       nan,       nan,
               nan]],

       [[      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
...
        [30.921225, 27.898603, 18.776686, ...,       nan,       nan,
               nan],
        [29.754301, 30.90201 , 26.216688, ...,       nan,       nan,
               nan],
        [27.801273, 30.98887 , 30.531118, ...,       nan,       nan,
               nan]],

       [[      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        [      nan,       nan,       nan, ...,       nan,       nan,
               nan],
        ...,
        [29.991274, 27.619225, 18.536316, ...,       nan,       nan,
               nan],
        [27.74555 , 29.49825 , 24.989594, ...,       nan,       nan,
               nan],
        [25.578426, 29.367172, 29.218924, ...,       nan,       nan,
               nan]]], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 240B 2024-09-01 2024-09-02 ... 2024-09-30
  * lat      (lat) float64 672B -34.75 -34.25 -33.75 -33.25 ... 5.75 6.25 6.75
  * lon      (lon) float64 672B -74.75 -74.25 -73.75 ... -34.25 -33.75 -33.25
Attributes:
    long_name:  daily maximum temperature [deg c] 
    units:      C

In [ ]:
# dataset com dias com tmax > p90
dias_acima_p90

<xarray.DataArray 'dias_acima_p90_setembro_2024' (lat: 84, lon: 84)> Size: 56kB
array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [16, 18, 18, ...,  0,  0,  0],
       [15, 19, 14, ...,  0,  0,  0],
       [14, 15, 10, ...,  0,  0,  0]])
Coordinates:
  * lat       (lat) float64 672B -34.75 -34.25 -33.75 -33.25 ... 5.75 6.25 6.75
  * lon       (lon) float64 672B -74.75 -74.25 -73.75 ... -34.25 -33.75 -33.25
    quantile  float64 8B 0.9
Attributes:
    long_name:  daily maximum temperature [deg c] 
    units:      C

In [ ]:
# dataset com dias com tmax > p90
dias_acima_p90

<xarray.DataArray 'dias_acima_p90_setembro_2024' (lat: 84, lon: 84)> Size: 56kB
array([[ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [16, 18, 18, ...,  0,  0,  0],
       [15, 19, 14, ...,  0,  0,  0],
       [14, 15, 10, ...,  0,  0,  0]])
Coordinates:
  * lat       (lat) float64 672B -34.75 -34.25 -33.75 -33.25 ... 5.75 6.25 6.75
  * lon       (lon) float64 672B -74.75 -74.25 -73.75 ... -34.25 -33.75 -33.25
    quantile  float64 8B 0.9
Attributes:
    long_name:  daily maximum temperature [deg c] 
    units:      C